## Goal Setting and Monitoring

### Overview
Goal Setting and Monitoring is about giving agents specific objectives to work towards and equipping them with the means to track their progress and determine if those objectives have been met. Planning typically involves an agent taking a high-level objective and autonomously, or semi-autonomously, generating a series of itermediate steps or sub-goals. These steps can then be executed sequentially or in a more complex flow, potentially involving other patterns like tool use, routing, or multi-agent collaboration. The planning mechanism might involve sophisticated search algorithms, logical reasoning, or increaingly, leveraging the capabilities of LLMs to generate plausible and effective plans based on their training data and understanding of tasks.

### Practical Applications & Use Cases
This pattern is essential for building agents that can operate autonomously and realiably in complex, real-world scenarios, such as:

- Customer Support Automation: An agent's goal might be to "resolve customer's billing inquiry". It monitors the conversation, checks database entries, and uses tools to adjust billing. Success is monitored by confirming the billing change and receiving positive customer feedback. If the issue isn't resolved, it escalates.

- Personalized Learning Systems: A learning agent might have the goal to "improve students' understanding of algebra". It monitors the student's progress on exercises, adapts teaching materials, and tracks performance metrics like accuracy and completion time, adjusting its approach if the student struggles.

- Project Management Assistants: An agent could be tasked with "ensuring project milestone X is completed by Y date". It moinmonitors task statuses, team communications, and resource availability, flagging delays and suggesting corrective actions if the goal is at risk.

- Automated Trading Bots: A trading agent's goal might be to "maximize portfolio gains while staying within risk tolerance". It continuously monitors market data, its current portfolio value, and risk indicators, executing trades when conditions alight with its goals and adjusting strategy of risk thresholds are breached.

- Robotics and Autonomous Vehicles: An autonomous vehicle's primary goal is "safety transport passengers from A to B". It constantly monitors its environment (other vehicles, pedestrians, traffic signals), its own state (speed, fuel), and its progressalong the planned route, adapting its driving behavior to achieve the goal safely and efficiently.

- Content Moderation: An agent's goal could be to "identify and remove harmful content from platform X". It monitors incoming content, applies calssification models, and tracks metrics like false positives/negatives, adjusting its filtering criteria or scalating ambiguous casess to human reviewers.

The pattern is fundamental for agents thatneed to operate reliably, achieve specific outcomes, and adapt to dynamic conditions, providing the necessary framework for intelligent self-management.

### Hands-On Code Example
This scipt shows an autonomous AI agent engineered to generate and refine Python code. Its core function is to produce solutions for specified problems, ensuring adherence to user-defined quality benchmarks.

In [ ]:
"""
- Accepts a coding problem (use case) in code or can be as input.
- Accepts a list of goals (e.g., "simple", "tested", "handles edge cases") in code or can be input.
- Uses an LLM to generate and refine Python code until the goals are met (with max 5 iterations).
- LLM answers with True/False to check if goals have been met.
- Saves the final code in a .py file with a clean filename and a header comment.
"""

import os
import random
import re
from dotenv import get_key
from pathlib import Path

from langchain_openai import ChatOpenAI

# --- Configuration
API_KEY = get_key("../.env", "OPENAI_API_KEY")

llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0.3,
    api_key=API_KEY
)


# --- Utility Functions
def generate_prompt(use_case: str, goals: list[str], previous_code: str="", feedback: str="") -> str:
    print("-- Constructing prompt for code generation...")
    
    base_prompt = f"""
    You are an AI coding agent. Your job is to write Python code based on the following use case:

    Use Case: {use_case}

    Your goals are: 
    {chr(10).join(f"- {g.strip()}" for g in goals)}
    """

    if previous_code:
        print("Adding previous code to the prompt for refinement.")
        base_prompt += f"\nPreviously generated code: \n{previous_code}"
    if feedback:
        print("Including feedback for revision.")
        base_prompt += f"\nFeedback on the previous version: \n{feedback}\n"

    base_prompt += "\nReturn only the revised Python code. Do not include comments or explanations outside the code."

    return base_prompt


def get_code_feedback(code: str, goals: list[str]) -> str:
    print("Evaluating code against the goals...")
    feedback_prompt = f"""
    You are a Python code reviewer. A code snippet is shown below. Based on the following goals:
    {chr(10).join(f"- {g.strip()}" for g in goals)}
    
    Please critique this code and identify if the goals are met. Mention if improvements are needed for clarity, simplicity, correctness, or test coverage.
    Code:
    {code}
    """
    return llm.invoke(feedback_prompt)


def goals_met(feedback_text: str, goals: list[str]) -> bool:
    """
    Uses the LLM to evaluate whether the goals have been met based on the feedback text.
    Returns True or False (parsed from LLM output).
    """
    review_prompt = f"""
    You are an AI reviewer.

    Here are the goals:
    {chr(10).join(f"- {g.strip()}" for g in goals)}

    Here's the feedback on the code:
    \"\"\"
    {feedback_text}
    \"\"\"

    Based on the feedback above, have the goals been met?
    Respond with only one word: True or False.
    """
    response = llm.invoke(review_prompt).content.strip().lower()
    
    return response == "true"


# test if this gets rid of indentation?
def clean_code_block(code: str) -> str:
    lines = code.strip().splitlines()
    if lines and lines[0].strip().startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


def add_comment_header(code: str, use_case: str) -> str:
    comment = f"# This Python program implements the following use case:\n# {use_case.strip()}\n"
    return comment + "\n" + code


def to_snake_case(text: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9 ]", "", text)
    return re.sub(r"\s+", "_", text.strip().lower())


def save_code_to_file(code: str, use_case: str) -> str:
    print("Saving final code to file...")

    summary_prompt = (
        f"Summarize the following use case into a single lowercasre word or phrase, "
        f"no more than 10 characters, suitable for a Python filename:\n\n{use_case}"
    )
    raw_summary = llm.invoke(summary_prompt).content.strip()
    short_name = re.sub(r"^a-zA-Z0-9_", "", raw_summary.replace(" ", "_").lower()[:10])

    random_suffix = str(random.randint(1000, 9999))
    filename = f"{short_name}_{random_suffix}.py"
    filepath = Path.cwd() / filename
    
    with open(filepath, "w") as f:
        f.write(code)

    print(f"Code saved to: {filepath}")
    return str(filepath)


def run_code_agent(use_case: str, goals_input: str, max_iterations: int=5) -> int:
    goals = [g.strip() for g in goals_input.split(",")]

    print(f"\nUse Case: {use_case}")
    print("\nGoals:")
    for g in goals:
        print(f"  - {g}")

    previous_code = ""
    feedback = ""

    for i in range(max_iterations):
        print(f"\n=== Iteration {i + 1} of {max_iterations} ===")
        prompt = generate_prompt(
            use_case,
            goals,
            previous_code,
            feedback if isinstance(feedback, str) else feedback.content
        )

        print("Generating Code...")
        code_response = llm.invoke(prompt)
        raw_code = code_response.content.strip()
        code = clean_code_block(raw_code)
        print("\n Generated Code:\n" + "-" * 50 + f"\n{code}\n" + "-" * 50)

        print("\nSubmitting Code for feedback review...")
        feedback = get_code_feedback(code, goals)
        feedback_text = feedback.content.strip()
        print("\nFeedback Received:\n" + "-" * 50 + f"\n{feedback_text}\n" + "_" * 50)

        if goals_met(feedback_text, goals):
            print("LLM confirms goals are met. Stopping iterations.")
            break

        print("Goals not fully met. Preparing for next iteration...")
        previous_code = code

    final_code = add_comment_header(code, use_case)
    return save_code_to_file(final_code, use_case)


if __name__ == "__main__":
    print("Welcome to the AI Code Generating Agent")

    # Example 1
    use_case_input = "Write code which takes a command line input of a word doc or docx file and opens it and counts the number of words, and characters in it and prints all"
    goals_input = "Code simple to understand, Functionally correct, Handles edge cases"
    run_code_agent(use_case_input, goals_input)

Welcome to the AI Code Generating Agent

Use Case: Write code which takes a command line input of a word doc or docx file and opens it and counts the number of words, and characters in it and prints all

Goals:
  - Code simple to understand
  - Functionally correct
  - Handles edge cases

=== Iteration 1 of 5 ===
-- Constructing prompt for code generation...
Generating Code...

 Generated Code:
--------------------------------------------------
import sys
import os
import subprocess
import tempfile
import shutil

def extract_text_from_docx(path):
    try:
        from docx import Document
    except Exception:
        raise RuntimeError("python-docx library is required to read .docx files. Install via 'pip install python-docx'.")
    doc = Document(path)
    parts = []
    for para in doc.paragraphs:
        if para.text:
            parts.append(para.text)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text:
     

### Key Takeaways
- Goal Setting and Monitoring equipts agents with purpose and mechanisms to track progress.

- Goals should be specific, measurable, achievable, relevant, and time-bound (SMART).

- Clearly defining metrics and success criteria is essential for effective monitoring.

- Monitoring involves observing agent actions, environmental states, and tool outputs.

- Feedback loops from monitoring allow agents to adapt, revise plans, or escalate issues.